In [84]:
import pandas as pd
import numpy as np

In [85]:
ball = pd.read_csv("Datasets/Cleaned_Datasets/ball_cleaned_data.csv")
matches = pd.read_csv("Datasets/Cleaned_Datasets/matches_cleaned_data.csv")

In [86]:
#DATE HANDLING
matches["match_date"] = pd.to_datetime(matches["match_date"], errors="coerce")

# Merge match_date and venue into ball data
ball = ball.merge(
    matches[["match_id", "match_date", "venue"]],
    on="match_id",
    how="left"
)

In [87]:
ball = ball.sort_values(
    by=["batter", "match_date", "match_id", "over_number", "ball_number"]
)

In [88]:
# PLAYER PER MATCH AGGREGATION
batting_match = (
    ball.groupby(
        ["match_id", "match_date", "batter", "team_batting", "team_bowling", "venue"]
    )
    .agg(
        runs_scored=("batter_runs", "sum"),
        balls_faced=("ball_number", "count"),
        fours=("batter_runs", lambda x: (x == 4).sum()),
        sixes=("batter_runs", lambda x: (x == 6).sum()),
        dismissed=("is_wicket", "max")
    )
    .reset_index()
)

In [89]:
# SORT FOR TEMPORAL FEATURES
batting_match = batting_match.sort_values(
    by=["batter", "match_date", "match_id"]
).reset_index(drop=True)

In [90]:
# PERFORMANCE METRICS
batting_match["strike_rate"] = np.where(
    batting_match["balls_faced"] > 0,
    (batting_match["runs_scored"] / batting_match["balls_faced"]) * 100,
    0
)

In [91]:
#  CAREER FEATURES
batting_match["career_matches"] = batting_match.groupby("batter").cumcount()
batting_match["career_matches"] = batting_match.groupby("batter")["career_matches"].shift(1).fillna(0).astype(int)

batting_match["career_runs"] = batting_match.groupby("batter")["runs_scored"].cumsum().shift(1).fillna(0)
batting_match["career_balls"] = batting_match.groupby("batter")["balls_faced"].cumsum().shift(1).fillna(0)

batting_match["career_avg_runs"] = np.where(
    batting_match["career_matches"] > 0,
    batting_match["career_runs"] / batting_match["career_matches"],
    0
)

batting_match["career_strike_rate"] = np.where(
    batting_match["career_balls"] > 0,
    (batting_match["career_runs"] / batting_match["career_balls"]) * 100,
    0
)

In [92]:
# RECENT FORM FEATURES (last 5 and 10 matches)

# Shifted series to avoid data leakage
batting_match["shifted_runs"] = batting_match.groupby("batter")["runs_scored"].shift(1)
batting_match["shifted_balls"] = batting_match.groupby("batter")["balls_faced"].shift(1)
batting_match["shifted_dismissed"] = batting_match.groupby("batter")["dismissed"].shift(1)

In [93]:
# Last 5 matches form
batting_match["form_runs_last_5"] = (
    batting_match.groupby("batter")["shifted_runs"]
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [94]:
# Last 10 matches form
batting_match["form_runs_last_10"] = (
    batting_match.groupby("batter")["shifted_runs"]
    .rolling(window=10, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [95]:
# Recent strike rate: last 5 matches
def calculate_recent_sr(runs_series, balls_series, window):
    total_runs = runs_series.rolling(window=window, min_periods=1).sum()
    total_balls = balls_series.rolling(window=window, min_periods=1).sum()
    return np.where(total_balls > 0, (total_runs / total_balls) * 100, 0)

batting_match["recent_sr_last_5"] = calculate_recent_sr(
    batting_match["shifted_runs"], batting_match["shifted_balls"], 5
)

In [96]:
# Recent dismissal rate
batting_match["recent_dismissal_rate_5"] = (
    batting_match.groupby("batter")["shifted_dismissed"]
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

In [97]:
# OPPONENT SPECIFIC FEATURE
def safe_expanding_mean(group):
    return group.shift(1).expanding().mean()

batting_match["avg_runs_vs_opponent"] = (
    batting_match.groupby(["batter", "team_bowling"])["runs_scored"]
    .transform(safe_expanding_mean)
    .fillna(0)
)

In [98]:
# VENUE-SPECIFIC FEATURE
batting_match["avg_runs_at_venue"] = (
    batting_match.groupby(["batter", "venue"])["runs_scored"]
    .transform(safe_expanding_mean)
    .fillna(0)
)

In [99]:
# BOUNDARY RATE
total_boundaries = (
    batting_match.groupby("batter")["fours"].cumsum().shift(1) +
    batting_match.groupby("batter")["sixes"].cumsum().shift(1)
).fillna(0)

batting_match["career_boundary_rate"] = np.where(
    batting_match["career_balls"] > 0,
    (total_boundaries / batting_match["career_balls"]) * 100,
    0
)

In [100]:
# TARGET VARIABLE
# Shift -1 to get NEXT match's runs
batting_match["target_next_match_runs"] = batting_match.groupby("batter")["runs_scored"].shift(-1)

# Also create a binary target for classification
batting_match["target_high_score"] = (batting_match["target_next_match_runs"] >= 30).astype(int)

# IMPORTANT: Remove rows where target is NaN (last match for each player)
batting_match = batting_match[batting_match["target_next_match_runs"].notna()].copy()

In [101]:
# FINAL FEATURE SET
drop_cols = ["runs_scored", "career_runs", "career_balls", 
             "shifted_runs", "shifted_balls", "shifted_dismissed"]
features_df = batting_match.drop(columns=drop_cols)

features_df = features_df.fillna(0)

In [102]:
# ENCODING 
from sklearn.preprocessing import LabelEncoder
import os
import pickle

In [103]:
# Frequency Encoding 
high_card_cols = ["batter", "team_batting", "team_bowling", "venue"]
frequency_encoders = {}
for col in high_card_cols:
    if col in features_df.columns:
        # Compute frequency map
        freq_map = features_df[col].value_counts(normalize=True)
        # Apply frequency encoding
        features_df[col + "_freq"] = features_df[col].map(freq_map)
        # Save the mapping for future use
        frequency_encoders[col] = freq_map.to_dict()

In [104]:
# Label encoding for small cardinality / binary
le = LabelEncoder()
features_df["dismissed_enc"] = le.fit_transform(features_df["dismissed"])

In [105]:
#  One Hot Encoding 
low_card_cols = []

for col in ["batting_style", "bowling_style", "opponent_team", "home_away"]:
    if col in features_df.columns:
        low_card_cols.append(col)

features_df = pd.get_dummies(
    features_df,
    columns=low_card_cols,
    drop_first=True
)

In [106]:
# Drop original categorical columns
features_df = features_df.drop(columns=["batter", "team_batting", "team_bowling", "venue", "dismissed"])

In [107]:
os.makedirs("Datasets/Final_Features", exist_ok=True)

features_df.to_csv(
    "Datasets/Final_Features/final_datasets.csv",
    index=False
)

In [108]:
features_df.dtypes

match_id                            int64
match_date                 datetime64[ns]
balls_faced                         int64
fours                               int64
sixes                               int64
strike_rate                       float64
career_matches                      int64
career_avg_runs                   float64
career_strike_rate                float64
form_runs_last_5                  float64
form_runs_last_10                 float64
recent_sr_last_5                  float64
recent_dismissal_rate_5           float64
avg_runs_vs_opponent              float64
avg_runs_at_venue                 float64
career_boundary_rate              float64
target_next_match_runs            float64
target_high_score                   int64
batter_freq                       float64
team_batting_freq                 float64
team_bowling_freq                 float64
venue_freq                        float64
dismissed_enc                       int64
dtype: object

In [109]:
# Save encoders
feature_pipeline = {
    "frequency_encoders": frequency_encoders,
    "dismissed_label_encoder": le,
    "final_feature_columns": features_df.columns.tolist()
}

with open("Datasets/Final_Features/feature_pipelines.pkl", "wb") as f:
    pickle.dump(feature_pipeline, f)

In [110]:
features_df.head(10)

,match_id,match_date,balls_faced,fours,sixes,strike_rate,career_matches,career_avg_runs,career_strike_rate,form_runs_last_5,...,avg_runs_vs_opponent,avg_runs_at_venue,career_boundary_rate,target_next_match_runs,target_high_score,batter_freq,team_batting_freq,team_bowling_freq,venue_freq,dismissed_enc
0,548346,2012-04-29,10,0,1,100.000000,0,0.000000,0.000000,0.00,...,0.0,0.000000,0.000000,3.0,0,0.001299,0.117272,0.121169,0.064187,1
1,548352,2012-05-04,3,0,0,100.000000,0,0.000000,100.000000,10.00,...,0.0,0.000000,10.000000,8.0,0,0.001299,0.117272,0.112371,0.041571,1
2,548359,2012-05-08,8,1,0,100.000000,1,13.000000,100.000000,6.50,...,0.0,0.000000,7.692308,10.0,0,0.001299,0.117272,0.109950,0.041157,1
3,548373,2012-05-18,4,2,0,250.000000,2,10.500000,100.000000,7.00,...,0.0,8.000000,9.523810,4.0,0,0.001299,0.117272,0.098849,0.041157,0
4,548376,2012-05-20,5,0,0,80.000000,3,10.333333,124.000000,7.75,...,0.0,9.000000,16.000000,7.0,0,0.001299,0.117272,0.114142,0.041157,1
5,598000,2013-04-05,4,1,0,175.000000,4,8.750000,116.666667,7.00,...,0.0,7.333333,13.333333,14.0,0,0.001299,0.117272,0.018482,0.041157,0
6,598004,2013-04-07,12,0,1,116.666667,5,8.400000,123.529412,6.40,...,4.0,7.250000,14.705882,3.0,0,0.001299,0.117272,0.114142,0.041157,1
7,598048,2013-04-09,4,0,0,75.000000,6,9.333333,121.739130,8.60,...,9.0,0.000000,13.043478,16.0,0,0.001299,0.117272,0.114142,0.054384,1
8,598010,2013-04-12,9,2,0,177.777778,7,8.428571,118.000000,7.60,...,0.0,0.000000,12.000000,4.0,0,0.001299,0.117272,0.113906,0.051668,1
9,598013,2013-04-14,5,0,0,80.000000,8,9.375000,127.118644,8.80,...,0.0,0.000000,13.559322,19.0,0,0.001299,0.117272,0.112430,0.065013,1
